## Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

## Load and Prepare Product Data

In [2]:
def load_products(filepath='cleaned_products.csv'):
    """Load cleaned product data from CSV."""
    df = pd.read_csv(filepath, encoding='utf-8')
    # Ensure required columns exist
    required_cols = ['title', 'description', 'features', 'brand', 'price_numeric', 'technical_details']
    for col in required_cols:
        if col not in df.columns:
            df[col] = ''  # fill missing with empty string
    # Fill NaN with empty strings
    df[required_cols] = df[required_cols].fillna('')
    return df

## Build a Unified Text Representation

In [3]:
def build_text_representation(row):
    """Combine product text fields into one string."""
    parts = []
    parts.append(str(row['title']))
    parts.append(str(row.get('description', '')))
    parts.append(str(row.get('features', '')))
    # Parse technical_details (JSON string) into key-value pairs
    tech = row.get('technical_details', '')
    if tech:
        try:
            tech_dict = json.loads(tech)
            # Convert dict to "key: value" strings
            tech_str = ' '.join([f"{k}: {v}" for k, v in tech_dict.items()])
            parts.append(tech_str)
        except:
            pass
    # Add brand
    parts.append(str(row.get('brand', '')))
    return ' '.join(parts).lower()

## Create TF‑IDF Matrix and Compute Similarity

In [4]:
def create_tfidf_matrix(texts):
    """Create TF-IDF matrix from list of texts."""
    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=10000)
    tfidf_matrix = vectorizer.fit_transform(texts)
    return vectorizer, tfidf_matrix

## Generate Similarity Reasons

In [5]:
def generate_reasons(query_row, candidate_row, common_words):
    """Generate human-readable reasons for similarity."""
    reasons = []
    # Brand
    q_brand = str(query_row.get('brand', '')).lower()
    c_brand = str(candidate_row.get('brand', '')).lower()
    if q_brand and q_brand == c_brand:
        reasons.append(f"Same brand: {c_brand.title()}")
    
    # Price similarity
    q_price = query_row.get('price_numeric')
    c_price = candidate_row.get('price_numeric')
    if pd.notna(q_price) and pd.notna(c_price) and q_price > 0:
        price_diff = abs(q_price - c_price) / q_price
        if price_diff <= 0.2:
            reasons.append("Similar price range")
    
    # Common keywords (from title/description overlap)
    if common_words:
        # Take top 3 common words (excluding very common stopwords)
        top_words = [w for w in common_words if len(w) > 3][:3]
        if top_words:
            reasons.append(f"Shared keywords: {', '.join(top_words)}")
    
    # Technical details overlap
    q_tech = query_row.get('technical_details', '')
    c_tech = candidate_row.get('technical_details', '')
    if q_tech and c_tech and q_tech == c_tech:
        reasons.append("Similar technical specifications")
    
    # Category (if available)
    q_cat = query_row.get('category', '')
    c_cat = candidate_row.get('category', '')
    if q_cat and q_cat == c_cat:
        reasons.append(f"Same category: {q_cat}")
    
    if not reasons:
        reasons.append("High overall text similarity")
    return reasons

## Query Function

In [9]:
def recommend_similar_products(query, products_df, vectorizer, tfidf_matrix, top_n=5):
    """
    Recommend similar products.
    query: either an ASIN (str) or a search term (str).
    Returns a DataFrame of top_n similar products with reasons.
    """
    # Define columns for the output DataFrame
    result_columns = ['asin', 'title', 'brand', 'price', 'similarity_score', 'reasons']
    
    # If query is an ASIN, get the corresponding product row
    query_idx = None
    query_row = None
    query_text = query.lower()  # default: treat as free text
    
    if 'asin' in products_df.columns and query in products_df['asin'].values:
        query_idx = products_df[products_df['asin'] == query].index[0]
        query_row = products_df.iloc[query_idx]
        query_text = query_row['text_representation']
    
    # Transform query to TF-IDF vector
    query_vec = vectorizer.transform([query_text])
    
    # Compute cosine similarity with all products
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Exclude the query product itself if it was an ASIN
    if query_idx is not None:
        sim_scores[query_idx] = -1.0  # ensure it's not selected
    
    # Get top indices (only positive similarities)
    top_indices = sim_scores.argsort()[::-1][:top_n]
    
    # Build result list
    results = []
    for idx in top_indices:
        if sim_scores[idx] <= 0:
            continue
        candidate = products_df.iloc[idx]
        
        # Determine common words for reason generation
        if query_row is not None:
            query_words = set(query_row['title'].lower().split()) | set(str(query_row.get('description', '')).lower().split())
        else:
            query_words = set(query_text.split())
        candidate_words = set(candidate['title'].lower().split()) | set(str(candidate.get('description', '')).lower().split())
        common_words = query_words.intersection(candidate_words)
        
        # Generate reasons
        if query_row is not None:
            reasons = generate_reasons(query_row, candidate, common_words)
        else:
            if common_words:
                reasons = ["Matches your search terms: " + ', '.join(list(common_words)[:3])]
            else:
                reasons = ["High overall text similarity"]
        
        results.append({
            'asin': candidate.get('asin', ''),
            'title': candidate.get('title', ''),
            'brand': candidate.get('brand', ''),
            'price': candidate.get('price_numeric', np.nan),
            'similarity_score': sim_scores[idx],
            'reasons': '; '.join(reasons)
        })
    
    # Always return a DataFrame with defined columns
    return pd.DataFrame(results, columns=result_columns)

## Full Example Usage

In [10]:
# Load data
products = load_products('/kaggle/input/datasets/tahmidakter/cleaned-products/cleaned_products_20260831_163948.csv')

# Build text representation
products['text_representation'] = products.apply(build_text_representation, axis=1)

# Create TF-IDF matrix
vectorizer, tfidf_matrix = create_tfidf_matrix(products['text_representation'].tolist())

# Example 1: Query by ASIN
asin_query = 'B083T4XJDY'  # Sennheiser HD 450BT
recommendations = recommend_similar_products(asin_query, products, vectorizer, tfidf_matrix, top_n=3)
print("Recommendations for ASIN", asin_query)
print(recommendations[['title', 'brand', 'price', 'similarity_score', 'reasons']])

# Example 2: Query by search term
search_query = 'wireless noise cancelling headphones'
recommendations = recommend_similar_products(search_query, products, vectorizer, tfidf_matrix, top_n=3)
print("\nRecommendations for search term:", search_query)
print(recommendations[['title', 'brand', 'price', 'similarity_score', 'reasons']])

Recommendations for ASIN B083T4XJDY
Empty DataFrame
Columns: [title, brand, price, similarity_score, reasons]
Index: []

Recommendations for search term: wireless noise cancelling headphones
                                               title  \
0  Active Noise Cancelling Ear Buds Wireless Earb...   
1  TOZO NC9 Hybrid Active Noise Cancelling Wirele...   
2  Wireless Earbuds, Bluetooth 5.4 Headphones Bas...   

                      brand  price  similarity_score  \
0  Visit the WHYKJTEK Store  79.99          0.316472   
1      Visit the TOZO Store  26.55          0.306715   
2   Visit the Btootos Store  23.99          0.291071   

                                             reasons  
0  Matches your search terms: cancelling, wireles...  
1  Matches your search terms: cancelling, noise, ...  
2  Matches your search terms: cancelling, wireles...  
